In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, '../src')
np.random.seed(42)

# Load configuration and prepare data (from notebook 01)
from data_loader import DataLoader
from drift_detectors import DriftDetector, MultiWindowDriftAnalysis

with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Recreate data splits (quick recap from 01_EDA)
loader = DataLoader(random_state=42)
df_synthetic, _ = loader.load_synthetic_data(
    n_samples_per_window=config['dataset']['synthetic']['n_samples_per_window'],
    n_features=config['dataset']['synthetic']['n_features'],
    n_windows=config['dataset']['synthetic']['n_windows'],
    drift_magnitude=config['dataset']['synthetic']['drift_magnitude'],
    drift_type=config['dataset']['synthetic']['drift_type']
)

windows = loader.split_into_windows(df_synthetic, n_windows=config['time_window']['n_windows'])
splits = loader.get_ref_and_test_splits(windows, ref_window_idx=0)
splits = loader.standardize_splits(splits)

print(f"✓ Data prepared: {len(splits)} windows, {splits[0]['X_ref'].shape[1]} features")


In [ ]:
# Initialize drift analysis
multi_window = MultiWindowDriftAnalysis(n_bins=config['drift_detection']['n_bins'])

# Compute drift metrics for all features and windows
drift_dfs = multi_window.compute_drift_matrix(
    splits,
    metrics=['psi', 'kl', 'js', 'wasserstein']
)

print("✓ Drift matrices computed for metrics:")
for metric, df in drift_dfs.items():
    print(f"\n{metric.upper()}:")
    print(f"  Shape: {df.shape} (features × windows)")
    print(f"  Range: [{df.min().min():.4f}, {df.max().max():.4f}]")
    print(f"  Mean: {df.mean().mean():.4f}")


In [ ]:
# Identify top drifting features
psi_df = drift_dfs['psi']
top_features_psi = multi_window.get_top_drifting_features(psi_df, top_n=5, aggregation='mean')

print("Top 5 Drifting Features (by PSI):")
print(top_features_psi)

# Get drift onset windows
threshold = config['drift_detection']['thresholds']['psi']['significant_drift']
onset_windows = multi_window.get_drift_onset_window(psi_df, threshold=threshold)

print(f"\nDrift Onset Windows (PSI > {threshold}):")
for feature, window in onset_windows.items():
    if window is not None:
        print(f"  {feature}: Window {window}")


In [ ]:
from visualizer import DriftVisualizer

visualizer = DriftVisualizer(output_dir='../reports/figures')

# Plot PSI heatmap
visualizer.plot_drift_heatmap(psi_df, metric_name='PSI', save_name='02_psi_heatmap.png')

# Plot KL heatmap
visualizer.plot_drift_heatmap(drift_dfs['kl'], metric_name='KL Divergence', save_name='02_kl_heatmap.png')

print("✓ Drift heatmaps saved")


## 4. Summary

**Drift Detection Results:**
- Computed 4 drift metrics across all features and windows
- Identified features with highest drift (PSI, KL divergence)
- Detected drift onset windows for significant drift
- Visualized drift patterns in heatmaps

**Key Findings:**
- Some features show gradual drift consistent with synthetic generation
- PSI metric ranges indicate varying degrees of drift
- Top drifting features can be prioritized for monitoring


## 3. Visualize Drift Heatmaps


## 2. Identify Top Drifting Features


## 1. Compute Drift Detection Metrics


# 02_drift_detection: Implementing and Computing Drift Metrics

This notebook implements multiple drift detection metrics (KL divergence, PSI, KS test) and computes them across all features and time windows.
